# 01 · Tiền xử lý dữ liệu

Bước tiền xử lý dữ liệu đi qua các bước sau:

1. **Nạp nhãn** từng bộ (`load_labels`): mỗi dòng là 1 segment `(path, start, end, label)` tính bằng giây.
2. **Gán split** train/val/test theo file split chính thức (`load_split`), phát hiện video nằm ở 2 split.
3. **Làm sạch có ghi nhận** (`clean_labels`): bỏ dòng thiếu trường, `end <= start`, trùng lặp, lớp mở rộng 10–15; mỗi bước ghi lại số dòng bị loại.
4. **Kiểm toán**: thống kê tổng quan, tính toàn vẹn split và thời gian, phân bố lớp, ma trận chuyển tiếp nhãn, ngữ cảnh trước segment `fallen`.

Kết quả thực tế: 24.933 segment trước làm sạch, còn 24.928 sau làm sạch (5 dòng bị loại, đều thuộc OF-Syn).

In [1]:
from __future__ import annotations
import sys
from pathlib import Path

# Notebook nằm trong thư mục notebooks/ -> thêm thư mục gốc repo vào sys.path
ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
print("Repo root:", ROOT)

Repo root: <repo>


**`tcfall/data.py` · phần import và hằng số**

Import và danh sách tên split. `LABEL_FILES` (trong config) ánh xạ tên bộ dữ liệu → file CSV nhãn gốc.

In [2]:
from __future__ import annotations
from dataclasses import dataclass, field
import numpy as np
import pandas as pd
from tcfall.config import LABEL_FILES, LABELS_RAW, NUM_CLASSES, SPLITS
SPLIT_NAMES = ("train", "val", "test")

**`tcfall/data.py` · load_labels**

**Bước 1 — nạp nhãn.** Đọc CSV nhãn gốc, thêm cột `dataset` để biết segment thuộc bộ nào và cột `dur = end - start` (độ dài segment, giây) dùng cho thống kê và lọc.

In [3]:
def load_labels(dataset: str) -> pd.DataFrame:
    df = pd.read_csv(LABELS_RAW / LABEL_FILES[dataset])
    df["dataset"] = dataset
    df["dur"] = df["end"] - df["start"]
    return df

**`tcfall/data.py` · load_split**

**Bước 2 — gán split.** Trả về dict `path → "train" | "val" | "test"`.
- OF-Syn dùng split `random` (đúng protocol OmniFall), các bộ khác dùng protocol `cs` (cross-subject).
- Nếu một video xuất hiện ở hai split khác nhau, đánh dấu `CONFLICT` để phát hiện rò rỉ dữ liệu.

In [4]:
def load_split(dataset: str, protocol: str = "cs") -> dict[str, str]:
    """path -> train/val/test. OF-Syn dùng split 'random' (theo protocol của OmniFall)."""
    base = SPLITS / "syn" / "random" if dataset == "of-syn" else SPLITS / protocol / dataset
    out = {}
    for s in SPLIT_NAMES:
        f = base / f"{s}.csv"
        if f.exists():
            for p in pd.read_csv(f)["path"]:
                if p in out and out[p] != s:
                    out[p] = "CONFLICT"
                else:
                    out[p] = s
    return out

**`tcfall/data.py` · CleanReport**

Lớp nhỏ ghi lại *audit trail*: danh sách `(tên bước, số dòng bị loại)`, để báo cáo minh bạch dữ liệu đã bị lọc thế nào.

In [5]:
@dataclass
class CleanReport:
    dataset: str
    steps: list[tuple[str, int]] = field(default_factory=list)

    def add(self, name: str, n_removed: int):
        self.steps.append((name, int(n_removed)))

**`tcfall/data.py` · clean_labels**

**Bước 3 — làm sạch.** Các bộ lọc chạy theo thứ tự, mỗi bộ ghi số dòng bị loại:
1. thiếu `path/label/start/end`;
2. `end <= start` (segment không hợp lệ);
3. trùng lặp hoàn toàn;
4. lớp mở rộng (nhãn ≥ 10), vì nhóm chỉ dùng 10 lớp lõi;
5. (tuỳ chọn) segment quá ngắn — mặc định **tắt** để không làm thay đổi tập test chuẩn.

Cuối cùng sắp xếp theo `(path, start)` để các bước sau có chuỗi thời gian đúng thứ tự.

In [6]:
def clean_labels(df: pd.DataFrame, min_dur: float = 0.0, keep_extended: bool = False
                 ) -> tuple[pd.DataFrame, CleanReport]:
    """Các bước lọc có thứ tự; mỗi bước ghi số dòng bị loại.

    min_dur mặc định 0: KHÔNG loại segment ngắn khi đánh giá (giữ nguyên test set chuẩn),
    chỉ dùng min_dur > 0 cho dữ liệu huấn luyện.
    """
    rep = CleanReport(df["dataset"].iloc[0] if len(df) else "?")
    n = len(df); df = df.dropna(subset=["path", "label", "start", "end"]); rep.add("thiếu trường", n - len(df))
    n = len(df); df = df[df["end"] > df["start"]]; rep.add("end <= start", n - len(df))
    n = len(df); df = df.drop_duplicates(subset=["path", "start", "end", "label"]); rep.add("trùng lặp", n - len(df))
    if not keep_extended:
        n = len(df); df = df[(df["label"] >= 0) & (df["label"] < NUM_CLASSES)]; rep.add("lớp mở rộng (10-15)", n - len(df))
    if min_dur > 0:
        n = len(df); df = df[df["dur"] >= min_dur]; rep.add(f"ngắn hơn {min_dur}s", n - len(df))
    return df.sort_values(["path", "start"]).reset_index(drop=True), rep

**`tcfall/data.py` · segment_sequences**

Gom segment theo video: `path → DataFrame` các segment theo thứ tự thời gian.

In [7]:
def segment_sequences(df: pd.DataFrame) -> dict[str, pd.DataFrame]:
    """path -> các segment của video đó theo thứ tự thời gian."""
    return {p: g.sort_values("start").reset_index(drop=True) for p, g in df.groupby("path", sort=False)}

**`tcfall/data.py` · transition_counts**

Đếm ma trận chuyển tiếp `M[a, b]` = số lần nhãn `a` được nối tiếp bởi nhãn `b` giữa hai segment liền kề trong cùng video.
Dùng để kiểm toán (trực quan hoá cấu trúc hành động → trạng thái) và ước lượng ma trận chuyển HMM.

In [8]:
def transition_counts(df: pd.DataFrame) -> np.ndarray:
    """Ma trận đếm chuyển tiếp nhãn giữa 2 segment liên tiếp trong cùng video."""
    M = np.zeros((NUM_CLASSES, NUM_CLASSES), dtype=int)
    for _, g in df.sort_values(["path", "start"]).groupby("path", sort=False):
        lab = g["label"].to_numpy()
        lab = lab[lab < NUM_CLASSES]
        for a, b in zip(lab[:-1], lab[1:]):
            M[a, b] += 1
    return M

**`tcfall/data.py` · transition_counts_timesampled**

Biến thể lấy mẫu theo thời gian: đọc nhãn mỗi `hop` giây thay vì theo ranh giới segment.
Khi chạy streaming không có ranh giới segment, nên ma trận này có **self-loop thực tế** (cùng nhãn lặp lại), khớp với cách hệ thống thật nhìn dữ liệu. Ma trận chuyển HMM cuối cùng được đếm theo cách này từ nhãn **train** của Staged + Syn (không dùng ItW, tránh rò rỉ).

In [9]:
def transition_counts_timesampled(df: pd.DataFrame, hop: float) -> np.ndarray:
    """Đếm chuyển tiếp giữa nhãn lấy mẫu mỗi `hop` giây (không dùng ranh giới segment) —
    đúng với cách pseudo-segment được tạo khi chạy streaming, nên có self-loop thực tế."""
    M = np.zeros((NUM_CLASSES, NUM_CLASSES), dtype=int)
    for _, g in df.sort_values(["path", "start"]).groupby("path", sort=False):
        st, en, lab = g["start"].to_numpy(), g["end"].to_numpy(), g["label"].to_numpy()
        times = np.arange(st.min(), en.max(), hop)
        idx = np.searchsorted(st, times, side="right") - 1
        ok = (idx >= 0) & (times < en[np.clip(idx, 0, None)])
        seq = lab[idx[ok]]
        seq = seq[seq < NUM_CLASSES]
        np.add.at(M, (seq[:-1], seq[1:]), 1)
    return M

**Ví dụ nhỏ:** chạy làm sạch và đếm chuyển tiếp trên dữ liệu giả, không cần tải dataset.

In [10]:
import pandas as pd
df = pd.DataFrame({
    "path":  ["v1", "v1", "v1", "v1", "v2", "v2"],
    "start": [0.0, 2.0, 3.0, 3.0, 0.0, 1.5],
    "end":   [2.0, 3.0, 8.0, 8.0, 1.5, 1.0],   # dòng 4 trùng lặp, dòng cuối end < start
    "label": [0, 1, 2, 2, 3, 12],
})
df["dataset"] = "demo"; df["dur"] = df.end - df.start
clean, rep = clean_labels(df)
print(rep.steps)
print(clean)
print(transition_counts(clean)[:3, :3])   # walk→fall, fall→fallen

[('thiếu trường', 0), ('end <= start', 1), ('trùng lặp', 1), ('lớp mở rộng (10-15)', 0)]
  path  start  end  label dataset  dur
0   v1    0.0  2.0      0    demo  2.0
1   v1    2.0  3.0      1    demo  1.0
2   v1    3.0  8.0      2    demo  5.0
3   v2    0.0  1.5      3    demo  1.5
[[0 1 0]
 [0 0 1]
 [0 0 0]]


## Kiểm toán dữ liệu (`pipeline/s01_audit_data.py`)

Sau khi có các hàm nạp và làm sạch, script bước 1 sinh các bảng kiểm toán vào `outputs_v2/data_audit/`. Các hàm vẽ hình (`fig_*`) được bỏ qua ở đây vì chỉ phục vụ báo cáo.

**`pipeline/s01_audit_data.py` · phần import và hằng số**

5 nguồn dữ liệu và miền tương ứng: OF-Syn (tổng hợp), 3 bộ dàn dựng (Staged), OOPS (ItW, video thực tế, dùng làm miền đích).

In [11]:
from __future__ import annotations
import json
import sys
from pathlib import Path
sys.path.insert(0, str(Path(ROOT / "pipeline/s01_audit_data.py").resolve().parent.parent))
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from tcfall.config import CLASS_NAMES, FALL, FALLEN, LIE_DOWN, LYING, NUM_CLASSES, OUTPUTS
from tcfall.data import clean_labels, load_labels, load_split, transition_counts
OUT = OUTPUTS / "data_audit"
DATASETS = ["of-syn", "gmdcsa24", "le2i", "caucafall", "OOPS"]
DOMAIN = {"of-syn": "Syn", "gmdcsa24": "Staged", "le2i": "Staged", "caucafall": "Staged", "OOPS": "ItW"}
plt.rcParams.update({"figure.dpi": 150, "savefig.dpi": 200, "font.size": 9})

**`pipeline/s01_audit_data.py` · overview**

Bảng tổng quan cho từng bộ: số video, số segment, tổng giờ, độ dài segment (trung bình/trung vị/max), số lớp có mặt, số segment thuộc lớp mở rộng.

In [12]:
def overview(raw: dict[str, pd.DataFrame]) -> pd.DataFrame:
    rows = []
    for d, df in raw.items():
        rows.append({
            "dataset": d, "domain": DOMAIN[d], "videos": df["path"].nunique(), "segments": len(df),
            "hours": df["dur"].sum() / 3600, "seg_dur_mean": df["dur"].mean(),
            "seg_dur_median": df["dur"].median(), "seg_dur_max": df["dur"].max(),
            "segs_per_video": len(df) / df["path"].nunique(),
            "classes_present": int(df.loc[df.label < NUM_CLASSES, "label"].nunique()),
            "extended_label_segments": int((df["label"] >= NUM_CLASSES).sum()),
        })
    return pd.DataFrame(rows)

**`pipeline/s01_audit_data.py` · split_integrity**

Kiểm tra toàn vẹn split: số video mỗi split, video có nhãn nhưng không thuộc split nào (ví dụ 3 video CAUCAFall), video có trong split mà không có nhãn, và **chồng lấn người quay** giữa train/test (nếu trùng người thì đánh giá sẽ lạc quan giả).

In [13]:
def split_integrity(raw) -> pd.DataFrame:
    rows = []
    for d, df in raw.items():
        sp = load_split(d)
        labeled = set(df["path"])
        in_split = set(sp)
        counts = pd.Series(list(sp.values())).value_counts()
        rows.append({
            "dataset": d, "train": int(counts.get("train", 0)), "val": int(counts.get("val", 0)),
            "test": int(counts.get("test", 0)), "conflict(path in >1 split)": int(counts.get("CONFLICT", 0)),
            "labeled_not_in_any_split": len(labeled - in_split),
            "split_entries_without_labels": len(in_split - labeled),
            "labeled_train": len([p for p in labeled if sp.get(p) == "train"]),
            "labeled_val": len([p for p in labeled if sp.get(p) == "val"]),
            "labeled_test": len([p for p in labeled if sp.get(p) == "test"]),
        })
        if d != "OOPS" and "subject" in df and (df["subject"] >= 0).any():
            subj = {s: set(df.loc[df.path.isin([p for p, v in sp.items() if v == s]), "subject"]) for s in ("train", "val", "test")}
            rows[-1]["subject_overlap_train_test"] = len(subj["train"] & subj["test"])
            rows[-1]["subject_overlap_train_val"] = len(subj["train"] & subj["val"])
    return pd.DataFrame(rows)

**`pipeline/s01_audit_data.py` · temporal_integrity**

Kiểm tra thời gian: các segment có chồng lên nhau, có khoảng trống, segment đầu có bắt đầu từ 0, số segment quá ngắn (< 0.25 s, < 0.5 s) hoặc quá dài (> 10 s).

In [14]:
def temporal_integrity(raw) -> pd.DataFrame:
    rows = []
    for d, df in raw.items():
        df = df.sort_values(["path", "start"])
        nxt_start = df.groupby("path")["start"].shift(-1)
        gap = nxt_start - df["end"]
        rows.append({
            "dataset": d,
            "overlap_pairs(>10ms)": int((gap < -0.01).sum()),
            "gap_pairs(>100ms)": int((gap > 0.1).sum()),
            "first_seg_not_at_0": int((df.groupby("path")["start"].min() > 0.1).sum()),
            "segments_<0.5s": int((df["dur"] < 0.5).sum()),
            "segments_<0.25s": int((df["dur"] < 0.25).sum()),
            "segments_>10s": int((df["dur"] > 10).sum()),
        })
    return pd.DataFrame(rows)

**`pipeline/s01_audit_data.py` · fallen_context**

Với mỗi segment `fallen` (và `lying`): segment ngay trước có phải `fall` (`lie_down`) không, và hành động chuyển tiếp gần nhất cách bao lâu.
Đây là **căn cứ dữ liệu** để chọn cửa sổ nhớ `W` và chu kỳ bán rã `τ` của luật chuyển trạng thái ở notebook 06.

In [15]:
def fallen_context(df: pd.DataFrame, dataset_label: str) -> dict:
    """Với mỗi segment fallen/lying: segment ngay trước là gì, và cách hành động chuyển tiếp
    tương ứng (fall/lie_down) gần nhất bao lâu — căn cứ dữ liệu cho cửa sổ W và chu kỳ bán rã τ."""
    res = {"domain": dataset_label}
    for static, trans, name in ((FALLEN, FALL, "fallen"), (LYING, LIE_DOWN, "lying")):
        prev_is_trans, delays, n = 0, [], 0
        for _, g in df.sort_values(["path", "start"]).groupby("path", sort=False):
            lab, st, en = g["label"].to_numpy(), g["start"].to_numpy(), g["end"].to_numpy()
            for i in np.where(lab == static)[0]:
                n += 1
                if i > 0 and lab[i - 1] == trans:
                    prev_is_trans += 1
                prior = np.where((lab[:i] == trans))[0]
                if len(prior):
                    delays.append(st[i] - st[prior[-1]])  # từ lúc BẮT ĐẦU transient gần nhất
        delays = np.array(delays)
        res.update({
            f"{name}_segments": n,
            f"{name}_prev_is_{CLASS_NAMES[trans]}_%": 100 * prev_is_trans / max(n, 1),
            f"{name}_has_{CLASS_NAMES[trans]}_before_%": 100 * len(delays) / max(n, 1),
            f"{name}_delay_median_s": float(np.median(delays)) if len(delays) else np.nan,
            f"{name}_delay_p90_s": float(np.percentile(delays, 90)) if len(delays) else np.nan,
            f"{name}_within_10s_%": 100 * float((delays <= 10).mean()) if len(delays) else np.nan,
        })
    return res

**`pipeline/s01_audit_data.py` · class_distribution**

Phân bố lớp theo nhóm: Staged, OF-Syn, ItW train/val/test — cho thấy lệch phân bố lớp giữa miền nguồn và miền đích (lý do có bước `prior_shift`).

In [16]:
def class_distribution(clean: dict[str, pd.DataFrame]) -> pd.DataFrame:
    groups = {}
    staged = pd.concat([clean[d] for d in ("gmdcsa24", "le2i", "caucafall")])
    groups["Staged (3 bộ)"] = staged
    groups["OF-Syn"] = clean["of-syn"]
    oops = clean["OOPS"]; sp = load_split("OOPS")
    for s in ("train", "val", "test"):
        groups[f"ItW {s}"] = oops[oops.path.map(sp) == s]
    rows = []
    for g, df in groups.items():
        c = df["label"].value_counts().reindex(range(NUM_CLASSES), fill_value=0)
        rows.append({"group": g, "segments": int(c.sum()), **{CLASS_NAMES[k]: int(c[k]) for k in range(NUM_CLASSES)}})
    return pd.DataFrame(rows)

**`pipeline/s01_audit_data.py` · main**

Chạy toàn bộ kiểm toán và lưu CSV. Cần dữ liệu trong `data/omnifall/` — gọi `main()` nếu đã tải dữ liệu.

In [17]:
def main():
    OUT.mkdir(parents=True, exist_ok=True)
    raw = {d: load_labels(d) for d in DATASETS}
    ov = overview(raw); ov.to_csv(OUT / "overview.csv", index=False)

    clean, rep_rows = {}, []
    for d, df in raw.items():
        c, rep = clean_labels(df)
        clean[d] = c
        rep_rows.append({"dataset": d, "rows_in": len(df), **{k: v for k, v in rep.steps}, "rows_out": len(c)})
    pd.DataFrame(rep_rows).to_csv(OUT / "cleaning.csv", index=False)

    split_integrity(raw).to_csv(OUT / "split_integrity.csv", index=False)
    temporal_integrity(raw).to_csv(OUT / "temporal_integrity.csv", index=False)

    cd = class_distribution(clean); cd.to_csv(OUT / "class_distribution.csv", index=False)
    fig_class_distribution(cd); fig_durations(clean)

    staged = pd.concat([clean[d] for d in ("gmdcsa24", "le2i", "caucafall")])
    mats = {"Staged": transition_counts(staged), "OF-Syn": transition_counts(clean["of-syn"]),
            "OF-ItW": transition_counts(clean["OOPS"])}
    for k, M in mats.items():
        pd.DataFrame(M, index=CLASS_NAMES, columns=CLASS_NAMES).to_csv(OUT / f"transition_{k}.csv")
    fig_transitions(mats)

    fc = pd.DataFrame([fallen_context(staged, "Staged"), fallen_context(clean["of-syn"], "OF-Syn"),
                       fallen_context(clean["OOPS"], "OF-ItW")])
    fc.to_csv(OUT / "fallen_context.csv", index=False)
    syn_metadata(raw["of-syn"]).to_csv(OUT / "syn_metadata.csv", index=False)

    with pd.option_context("display.width", 200, "display.max_columns", 30):
        for f in ("overview", "cleaning", "split_integrity", "temporal_integrity", "class_distribution", "fallen_context"):
            print(f"\n=== {f} ===\n", pd.read_csv(OUT / f"{f}.csv").round(2).to_string(index=False))
    print(f"\nĐã lưu vào {OUT}")

## ▶ Chạy thật trên dữ liệu thật

Phần trên định nghĩa code. Từ đây là các ô chạy thật bằng dữ liệu, checkpoint và GPU của dự án.

Chạy toàn bộ bước kiểm toán nhãn trên 5 bộ dữ liệu thật. Khác `main()`, phần này chỉ in ra, không ghi đè file kết quả.

In [18]:
import matplotlib.pyplot as plt

raw = {d: load_labels(d) for d in DATASETS}
print(overview(raw).round(2).to_string(index=False))
clean, rows = {}, []
for d, df in raw.items():
    c, rep = clean_labels(df)
    clean[d] = c
    rows.append({"dataset": d, "rows_in": len(df), **dict(rep.steps), "rows_out": len(c)})
print("\nLàm sạch nhãn:\n", pd.DataFrame(rows).to_string(index=False))
print("\nTính toàn vẹn split:\n", split_integrity(raw).to_string(index=False))
print("\nTính toàn vẹn thời gian:\n", temporal_integrity(raw).to_string(index=False))

  dataset domain  videos  segments  hours  seg_dur_mean  seg_dur_median  seg_dur_max  segs_per_video  classes_present  extended_label_segments
   of-syn    Syn   12000     19228  16.88          3.16            3.03         5.06            1.60               10                        0
 gmdcsa24 Staged     160       458   0.36          2.80            2.10        14.34            2.86               10                        0
     le2i Staged     190       967   0.79          2.95            2.25        51.32            5.09                9                        0
caucafall Staged     100       258   0.28          3.85            2.60        13.30            2.58               10                        0
     OOPS    ItW     818      4022   2.65          2.38            1.65        44.72            4.92               10                        0

Làm sạch nhãn:
   dataset  rows_in  thiếu trường  end <= start  trùng lặp  lớp mở rộng (10-15)  rows_out
   of-syn    19228             0    

Phân bố lớp, ngữ cảnh của "đã ngã", và ma trận chuyển nhãn thật của video ngoài đời.

In [19]:
print(class_distribution(clean).round(2).to_string(index=False))
staged = pd.concat([clean[d] for d in ("gmdcsa24", "le2i", "caucafall")])
print("\nNgữ cảnh đã ngã:\n", pd.DataFrame([fallen_context(staged, "Staged"), fallen_context(clean["of-syn"], "OF-Syn"), fallen_context(clean["OOPS"], "OF-ItW")]).round(2).to_string(index=False))
M = transition_counts(clean["OOPS"])
fig, ax = plt.subplots(figsize=(6, 5)); ax.imshow(M, cmap="Blues")
ax.set_xticks(range(10)); ax.set_xticklabels(CLASS_NAMES, rotation=60, ha="right"); ax.set_yticks(range(10)); ax.set_yticklabels(CLASS_NAMES)
for i in range(10):
    for j in range(10):
        if M[i, j]:
            ax.text(j, i, int(M[i, j]), ha="center", va="center", fontsize=7)
ax.set_title("OF-ItW: số lần chuyển từ nhãn hàng sang nhãn cột"); plt.tight_layout(); plt.show()

        group  segments  walk  fall  fallen  sit_down  sitting  lie_down  lying  stand_up  standing  other
Staged (3 bộ)      1683   376   258     248        84      207        15     15       141       106    233
       OF-Syn     19223  1940  2145    1449      1324     2708      1259   2344      1698      2432   1924
    ItW train       809    93   184     100         0       21         3      3        76        67    262
      ItW val       409    50    94      46         6       13         2      0        36        43    119
     ItW test      2804   324   636     318         9       70        11     17       253       264    902



Ngữ cảnh đã ngã:
 domain  fallen_segments  fallen_prev_is_fall_%  fallen_has_fall_before_%  fallen_delay_median_s  fallen_delay_p90_s  fallen_within_10s_%  lying_segments  lying_prev_is_lie_down_%  lying_has_lie_down_before_%  lying_delay_median_s  lying_delay_p90_s  lying_within_10s_%
Staged              248                 100.00                    100.00                   2.06                3.00               100.00              15                     73.33                        73.33                  3.55               5.52               100.0
OF-Syn             1449                  87.58                     87.92                   1.42                3.03               100.00            2344                     44.75                        44.80                  2.45               3.92               100.0
OF-ItW              464                  96.55                     97.63                   1.48                2.63                99.78              20                     2